# TATR Table Extraction Evaluation

This notebook runs a full TATR (Table Transformer) table-structure evaluation pipeline aligned with `test_surya.ipynb` for fair comparison:
- load dataset samples from Hugging Face
- run TATR structure inference
- map outputs to the 6 competition classes
- evaluate with `torchmetrics` mAP/mAR + IoU=0.50 precision/recall/F1
- save metrics and predictions for review.

In [1]:
from __future__ import annotations

## Environment Setup

Install dependencies in Colab with `uv`. This notebook uses Hugging Face `transformers` loading for TATR exactly like `Using_Table_Transformer_for_table_detection_and_table_structure_recognition.ipynb` (feature extractor/image processor + `TableTransformerForObjectDetection.from_pretrained(...)`).

In [2]:
import os

IN_COLAB = "COLAB_GPU" in os.environ
print(f"In Colab runtime: {IN_COLAB}")

In Colab runtime: True


In [3]:
!pip install uv
!uv pip install "torch" "torchvision"
!uv pip install "transformers==4.56.1" "datasets" "torchmetrics" "huggingface_hub" "pandas" "tqdm" "pillow" "numpy" "timm" "surya-ocr" "ultralytics"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 81.2 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.12 environment at: /usr
Audited 2 packages in 132ms
Using Python 3.12.12 environment at: /usr
Resolved 97 packages in 947ms                                        
Prepared 17 packages in 1.56s                                            
Uninstalled 4 packages in 382ms
Installed 17 packages in 88ms                               
 + cfgv==3.5.0
 + distlib==0.4.0
 + filetype==1.2.0
 - huggingface-hub==1.4.1
 + huggingface-hub==0.36.2
 + identify==2.6.16
 + lightning-utilities==0.15.2
 + nodeenv==1.10.0
 - opencv-python-headless==4.13.0.92
 + opencv-python-headless==4.11.0.86
 - pillow==11.3.0
 + pillow==10.4.0
 + pre-commit==4.5.1
 + pypdfium2==4.30.0
 + surya-ocr==0.17.1
 + torchmetrics==1.8.2
 - transformers==5.0.0
 + transformers==4.56.1
 + ultralytics==8.4.14
 + ultralytics-thop==2.0.18
 + virtualenv==20.38.0


## Imports

Imports are grouped by purpose: core utilities, dataset loading, model inference, metrics, and exports.

In [4]:
import json
import math
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from PIL import Image, ImageDraw
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm.auto import tqdm
from transformers import TableTransformerForObjectDetection

try:
    from transformers import DetrImageProcessor as DetrFeatureExtractor
except Exception:
    from transformers import DetrFeatureExtractor

## Configuration

Tune runtime behavior from `EvalConfig` (dataset selection, TATR model/inference settings, output paths, and metric thresholds).

In [5]:
@dataclass
class EvalConfig:
    # Dataset settings
    dataset_name: str = "ysif9/test-dataset"
    split: str = "train"

    # Sampling / runtime
    max_samples: Optional[int] = None
    random_seed: int = 42

    # Table detection backend: "tatr" | "surya_layout" | "yolo" | "none"
    detection_backend: str = "tatr"

    # TATR detection settings (used when detection_backend == "tatr")
    detection_model_name: str = "microsoft/table-transformer-detection"
    detection_score_threshold: float = 0.8
    max_tables_per_image: int = 1

    # Surya layout settings (used when detection_backend == "surya_layout")
    include_table_of_contents_as_table: bool = False
    layout_batch_size: Optional[int] = None

    # YOLO detection settings (used when detection_backend == "yolo")
    yolo_model_path: str = "best.pt"
    yolo_conf_threshold: float = 0.25
    yolo_table_class_names: Tuple[str, ...] = ("table",)

    # Structure model settings
    structure_model_name: str = "microsoft/table-transformer-structure-recognition-v1.1-all"
    device: Optional[str] = None
    batch_size: int = 8
    score_threshold: float = 0.5

    # Inference chunking
    inference_chunk_size: int = 100
    clear_cuda_cache_between_chunks: bool = True

    # Prediction cleanup
    dedupe_predictions: bool = True
    dedupe_iou_threshold: float = 0.98

    # Optional geometric filtering by detected table region
    keep_only_inside_best_table: bool = False
    min_inside_iob: float = 0.7

    # Evaluation
    map_thresholds: Tuple[float, ...] = (
        0.50,
        0.55,
        0.60,
        0.65,
        0.70,
        0.75,
        0.80,
        0.85,
        0.90,
        0.95,
    )

    # Output
    output_dir: str = "outputs/tatr_table_eval_full"
    save_visualizations: bool = False
    num_visualizations: int = 20


CONFIG = EvalConfig()

## Data Structures

Core dataclasses and geometry helpers shared across loading, inference mapping, evaluation, and export.

In [6]:
CLASS_NAMES: List[str] = [
    "table",
    "table column",
    "table row",
    "table column header",
    "table projected row header",
    "table spanning cell",
]
CLASS_NAME_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}


@dataclass
class Sample:
    image_id: int
    file_name: str
    image: Image.Image
    gt_boxes_xyxy: List[List[float]]
    gt_labels: List[int]


@dataclass
class Prediction:
    image_id: int
    bbox_xyxy: List[float]
    category_id: int
    score: float
    source: str = ""


def coco_xywh_to_xyxy(box: Sequence[float]) -> List[float]:
    x, y, w, h = [float(v) for v in box]
    return [x, y, x + w, y + h]


def xyxy_to_coco_xywh(box: Sequence[float]) -> List[float]:
    x1, y1, x2, y2 = [float(v) for v in box]
    return [x1, y1, max(0.0, x2 - x1), max(0.0, y2 - y1)]


def clip_box_xyxy(box: Sequence[float], width: int, height: int) -> List[float]:
    x1, y1, x2, y2 = [float(v) for v in box]
    x1 = min(max(0.0, x1), float(width))
    y1 = min(max(0.0, y1), float(height))
    x2 = min(max(0.0, x2), float(width))
    y2 = min(max(0.0, y2), float(height))
    if x2 < x1:
        x1, x2 = x2, x1
    if y2 < y1:
        y1, y2 = y2, y1
    return [x1, y1, x2, y2]


def box_iou_xyxy(a: Sequence[float], b: Sequence[float]) -> float:
    ax1, ay1, ax2, ay2 = [float(v) for v in a]
    bx1, by1, bx2, by2 = [float(v) for v in b]
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    denom = area_a + area_b - inter_area
    if denom <= 0:
        return 0.0
    return inter_area / denom

## Dataset Loading

Use the same `load_samples()` approach as `test_surya.ipynb`, including COCO `xywh` to `xyxy` conversion.

In [7]:
def load_samples(config: EvalConfig) -> List[Sample]:
    ds = load_dataset(config.dataset_name, split=config.split)
    if config.max_samples is not None:
        ds = ds.select(range(min(config.max_samples, len(ds))))

    samples: List[Sample] = []
    for idx, row in enumerate(tqdm(ds, desc="Loading HF dataset")):
        image = row["image"].convert("RGB")
        objects = row["objects"]
        boxes_xyxy = [coco_xywh_to_xyxy(b) for b in objects["bbox"]]
        labels = [int(c) for c in objects["category"]]
        file_name = row.get("file_name", f"image_{idx:06d}.png")
        samples.append(
            Sample(
                image_id=idx,
                file_name=file_name,
                image=image,
                gt_boxes_xyxy=boxes_xyxy,
                gt_labels=labels,
            )
        )
    return samples

## TATR Prediction Mapping

This section supports three table-detection backends before structure recognition:
- `tatr`: use `microsoft/table-transformer-detection`
- `surya_layout`: use Surya layout detector (`Table` regions)
- `none`: skip detection and run structure on the full page

After table-region selection, TATR structure recognition is applied to each crop, then mapped to the 6 competition classes.

In [8]:
def ensure_transformers_surya_compat() -> None:
    try:
        import torch
        from transformers import pytorch_utils as pu
    except Exception:
        return

    try:
        from transformers import modeling_utils as mu
    except Exception:
        mu = None

    if not hasattr(pu, "find_pruneable_heads_and_indices") and mu is not None and hasattr(mu, "find_pruneable_heads_and_indices"):
        pu.find_pruneable_heads_and_indices = mu.find_pruneable_heads_and_indices

    if not hasattr(pu, "prune_linear_layer") and mu is not None and hasattr(mu, "prune_linear_layer"):
        pu.prune_linear_layer = mu.prune_linear_layer

    if not hasattr(pu, "meshgrid"):
        pu.meshgrid = lambda *tensors, indexing="ij": torch.meshgrid(*tensors, indexing=indexing)


def _fix_processor_size_config(processor: Any) -> Any:
    size = getattr(processor, "size", None)
    if isinstance(size, dict) and "longest_edge" in size and "shortest_edge" not in size and "height" not in size:
        longest = int(size["longest_edge"])
        shortest = 800 if longest >= 800 else longest
        processor.size = {"shortest_edge": shortest, "longest_edge": longest}
    return processor


def _load_processor(model_name: str) -> Any:
    try:
        p = DetrFeatureExtractor.from_pretrained(model_name)
    except Exception:
        p = DetrFeatureExtractor()
    return _fix_processor_size_config(p)


def _init_surya_layout_predictor(config: EvalConfig) -> Any:
    ensure_transformers_surya_compat()
    from surya.settings import settings
    from surya.foundation import FoundationPredictor
    from surya.layout import LayoutPredictor

    foundation = FoundationPredictor(
        checkpoint=settings.LAYOUT_MODEL_CHECKPOINT,
        device=config.device or settings.TORCH_DEVICE_MODEL,
    )
    return LayoutPredictor(foundation)


def _layout_boxes_to_table_bboxes(layout_result: Any, include_toc: bool) -> List[List[float]]:
    keep_labels = {"Table"}
    if include_toc:
        keep_labels.add("TableOfContents")
    return [box.bbox for box in layout_result.bboxes if box.label in keep_labels]


def _init_yolo_detector(config: EvalConfig) -> Any:
    from ultralytics import YOLO
    return YOLO(config.yolo_model_path)


def _select_table_boxes_from_yolo(yolo_result: Any, config: EvalConfig) -> List[List[float]]:
    boxes_obj = getattr(yolo_result, "boxes", None)
    if boxes_obj is None or boxes_obj.xyxy is None:
        return []

    xyxy = boxes_obj.xyxy.detach().cpu().tolist()
    confs = boxes_obj.conf.detach().cpu().tolist() if boxes_obj.conf is not None else [1.0] * len(xyxy)
    classes = boxes_obj.cls.detach().cpu().tolist() if boxes_obj.cls is not None else [0] * len(xyxy)
    names = getattr(yolo_result, "names", {}) or {}

    wanted = {n.lower() for n in config.yolo_table_class_names}
    selected: List[Tuple[float, List[float]]] = []
    fallback: List[Tuple[float, List[float]]] = []

    for box, conf, cls_id in zip(xyxy, confs, classes):
        score = float(conf)
        if score < float(config.yolo_conf_threshold):
            continue
        cls_name = str(names.get(int(cls_id), cls_id)).lower()
        item = (score, [float(v) for v in box])
        fallback.append(item)
        if cls_name in wanted:
            selected.append(item)

    picks = selected if selected else fallback
    picks.sort(key=lambda x: x[0], reverse=True)
    return [b for _, b in picks[: max(1, int(config.max_tables_per_image))]]


def load_tatr(config: EvalConfig) -> Dict[str, Any]:
    device = config.device or ("cuda" if torch.cuda.is_available() else "cpu")

    structure_processor = _load_processor(config.structure_model_name)
    structure_model = TableTransformerForObjectDetection.from_pretrained(
        config.structure_model_name,
        low_cpu_mem_usage=False,
    ).to(device).eval()

    detection_processor = None
    detection_model = None
    layout_predictor = None
    yolo_detector = None

    if config.detection_backend == "tatr":
        detection_processor = _load_processor(config.detection_model_name)
        detection_model = TableTransformerForObjectDetection.from_pretrained(
            config.detection_model_name,
            low_cpu_mem_usage=False,
        ).to(device).eval()
    elif config.detection_backend == "surya_layout":
        layout_predictor = _init_surya_layout_predictor(config)
    elif config.detection_backend == "yolo":
        yolo_detector = _init_yolo_detector(config)

    return {
        "device": device,
        "structure_processor": structure_processor,
        "structure_model": structure_model,
        "detection_processor": detection_processor,
        "detection_model": detection_model,
        "layout_predictor": layout_predictor,
        "yolo_detector": yolo_detector,
    }


def _post_process_outputs(outputs: Any, img_sizes: Sequence[Tuple[int, int]], id2label: Dict[int, str], processor: Any, score_threshold: float) -> List[List[Dict[str, Any]]]:
    target_sizes = torch.tensor([(int(h), int(w)) for (w, h) in img_sizes], dtype=torch.int64)
    results = processor.post_process_object_detection(outputs, threshold=float(score_threshold), target_sizes=target_sizes)
    batch_objects: List[List[Dict[str, Any]]] = []
    for result in results:
        objects: List[Dict[str, Any]] = []
        scores = result["scores"].detach().cpu().tolist()
        labels = result["labels"].detach().cpu().tolist()
        boxes = result["boxes"].detach().cpu().tolist()
        for score, label, bbox in zip(scores, labels, boxes):
            objects.append({"label": id2label.get(int(label), str(label)), "score": float(score), "bbox": [float(v) for v in bbox]})
        batch_objects.append(objects)
    return batch_objects


def _infer_model_batch(model: TableTransformerForObjectDetection, processor: Any, images: Sequence[Image.Image], batch_size: int, score_threshold: float) -> List[List[Dict[str, Any]]]:
    device = next(model.parameters()).device
    all_objects: List[List[Dict[str, Any]]] = []
    for i in range(0, len(images), max(1, int(batch_size))):
        batch_images = images[i : i + max(1, int(batch_size))]
        encoding = processor(images=batch_images, return_tensors="pt")
        kwargs: Dict[str, Any] = {"pixel_values": encoding["pixel_values"].to(device)}
        if "pixel_mask" in encoding:
            kwargs["pixel_mask"] = encoding["pixel_mask"].to(device)
        with torch.no_grad():
            outputs = model(**kwargs)
        id2label = dict(model.config.id2label)
        all_objects.extend(_post_process_outputs(outputs, [img.size for img in batch_images], id2label, processor, score_threshold))
    return all_objects


def _select_table_boxes(detection_objects: List[Dict[str, Any]], max_tables: int) -> List[List[float]]:
    table_like = [obj for obj in detection_objects if normalize_tatr_label(str(obj.get("label", ""))) in {"table", "table rotated"}]
    table_like.sort(key=lambda x: float(x.get("score", 0.0)), reverse=True)
    return [obj["bbox"] for obj in table_like[: max(1, int(max_tables))]]


def normalize_tatr_label(label: str) -> str:
    return " ".join(label.lower().replace("_", " ").split())


LABEL_MAP: Dict[str, str] = {
    "table": "table",
    "table column": "table column",
    "table row": "table row",
    "table column header": "table column header",
    "table projected row header": "table projected row header",
    "table spanning cell": "table spanning cell",
    "column": "table column",
    "row": "table row",
    "column header": "table column header",
    "projected row header": "table projected row header",
    "spanning cell": "table spanning cell",
}


def map_tatr_label_to_class_id(label: str) -> Optional[int]:
    canonical = LABEL_MAP.get(normalize_tatr_label(label))
    if canonical is None:
        return None
    return CLASS_NAME_TO_ID[canonical]


def _deduplicate_predictions(predictions: List[Prediction], iou_threshold: float) -> List[Prediction]:
    deduped: List[Prediction] = []
    for class_id in sorted({p.category_id for p in predictions}):
        class_preds = [p for p in predictions if p.category_id == class_id]
        class_preds.sort(key=lambda p: float(p.score), reverse=True)
        kept: List[Prediction] = []
        for pred in class_preds:
            if not any(box_iou_xyxy(pred.bbox_xyxy, k.bbox_xyxy) >= iou_threshold for k in kept):
                kept.append(pred)
        deduped.extend(kept)
    return deduped


def _intersection_area(a: Sequence[float], b: Sequence[float]) -> float:
    ax1, ay1, ax2, ay2 = [float(v) for v in a]
    bx1, by1, bx2, by2 = [float(v) for v in b]
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    return max(0.0, inter_x2 - inter_x1) * max(0.0, inter_y2 - inter_y1)


def _box_area(box: Sequence[float]) -> float:
    x1, y1, x2, y2 = [float(v) for v in box]
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def _filter_inside_best_table(preds: List[Prediction], min_inside_iob: float) -> List[Prediction]:
    table_class_id = CLASS_NAME_TO_ID["table"]
    table_preds = [p for p in preds if p.category_id == table_class_id]
    if not table_preds:
        return preds
    best_table = max(table_preds, key=lambda p: p.score)
    filtered: List[Prediction] = []
    for p in preds:
        if p.category_id == table_class_id:
            filtered.append(p)
            continue
        inter = _intersection_area(p.bbox_xyxy, best_table.bbox_xyxy)
        pred_area = _box_area(p.bbox_xyxy)
        iob = inter / pred_area if pred_area > 0 else 0.0
        if iob >= min_inside_iob:
            filtered.append(p)
    return filtered


def map_tatr_objects_to_predictions(image_id: int, image_size: Tuple[int, int], structure_objects: List[Dict[str, Any]], config: EvalConfig, offset_x: float = 0.0, offset_y: float = 0.0) -> List[Prediction]:
    width, height = image_size
    preds: List[Prediction] = []
    for obj in structure_objects:
        class_id = map_tatr_label_to_class_id(str(obj.get("label", "")))
        if class_id is None:
            continue
        score = float(obj.get("score", 0.0))
        box = [float(obj.get("bbox", [0, 0, 0, 0])[0]) + offset_x, float(obj.get("bbox", [0, 0, 0, 0])[1]) + offset_y, float(obj.get("bbox", [0, 0, 0, 0])[2]) + offset_x, float(obj.get("bbox", [0, 0, 0, 0])[3]) + offset_y]
        box = clip_box_xyxy(box, width, height)
        preds.append(Prediction(image_id=image_id, bbox_xyxy=box, category_id=class_id, score=score, source=f"tatr/{normalize_tatr_label(str(obj.get('label', '')))}"))
    if config.keep_only_inside_best_table:
        preds = _filter_inside_best_table(preds, min_inside_iob=float(config.min_inside_iob))
    if config.dedupe_predictions:
        preds = _deduplicate_predictions(preds, iou_threshold=float(config.dedupe_iou_threshold))
    return preds


def run_tatr_inference(samples: List[Sample], config: EvalConfig, model_bundle: Dict[str, Any]) -> Dict[int, List[Prediction]]:
    import gc

    predictions_by_image: Dict[int, List[Prediction]] = {s.image_id: [] for s in samples}
    structure_model = model_bundle["structure_model"]
    structure_processor = model_bundle["structure_processor"]
    detection_model = model_bundle["detection_model"]
    detection_processor = model_bundle["detection_processor"]
    layout_predictor = model_bundle["layout_predictor"]
    yolo_detector = model_bundle.get("yolo_detector")

    chunk_size = max(1, int(config.inference_chunk_size))
    for chunk_start in tqdm(range(0, len(samples), chunk_size), desc="Inference chunks"):
        chunk_samples = samples[chunk_start : chunk_start + chunk_size]
        for sample in chunk_samples:
            per_image_preds: List[Prediction] = []

            if config.detection_backend == "tatr" and detection_model is not None and detection_processor is not None:
                det_objs_batch = _infer_model_batch(detection_model, detection_processor, [sample.image], 1, config.detection_score_threshold)
                table_boxes = _select_table_boxes(det_objs_batch[0], max_tables=config.max_tables_per_image)

            elif config.detection_backend == "surya_layout" and layout_predictor is not None:
                layout_res = layout_predictor([sample.image], batch_size=config.layout_batch_size)[0]
                table_boxes = _layout_boxes_to_table_bboxes(layout_res, include_toc=config.include_table_of_contents_as_table)
                table_boxes = [clip_box_xyxy(b, sample.image.size[0], sample.image.size[1]) for b in table_boxes]
                if config.max_tables_per_image is not None:
                    table_boxes = table_boxes[: max(1, int(config.max_tables_per_image))]

            elif config.detection_backend == "yolo" and yolo_detector is not None:
                yolo_results = yolo_detector.predict(sample.image, conf=float(config.yolo_conf_threshold), verbose=False)
                table_boxes = _select_table_boxes_from_yolo(yolo_results[0], config) if yolo_results else []

            else:
                table_boxes = [[0.0, 0.0, float(sample.image.size[0]), float(sample.image.size[1])]]

            if not table_boxes:
                predictions_by_image[sample.image_id] = []
                continue

            for table_box in table_boxes:
                x1, y1, x2, y2 = [int(round(v)) for v in table_box]
                x1 = max(0, min(x1, sample.image.size[0] - 1))
                y1 = max(0, min(y1, sample.image.size[1] - 1))
                x2 = max(x1 + 1, min(x2, sample.image.size[0]))
                y2 = max(y1 + 1, min(y2, sample.image.size[1]))
                crop = sample.image.crop((x1, y1, x2, y2))

                str_objs_batch = _infer_model_batch(structure_model, structure_processor, [crop], 1, config.score_threshold)
                mapped_preds = map_tatr_objects_to_predictions(sample.image_id, sample.image.size, str_objs_batch[0], config, float(x1), float(y1))
                per_image_preds.extend(mapped_preds)

            if config.dedupe_predictions:
                per_image_preds = _deduplicate_predictions(per_image_preds, iou_threshold=float(config.dedupe_iou_threshold))

            predictions_by_image[sample.image_id] = per_image_preds

        gc.collect()
        if config.clear_cuda_cache_between_chunks and torch.cuda.is_available():
            torch.cuda.empty_cache()

    return predictions_by_image

## Evaluation

This section computes:
- COCO-style detection metrics with `MeanAveragePrecision`
- detection-level `precision`, `recall`, and `f1` at IoU=0.50 from TP/FP/FN counts
- per-class `precision`, `recall`, and `f1` at IoU=0.50 for easier analysis.

In [9]:
# Legacy hotfix cell intentionally left blank.
# `load_tatr` and processor-size compatibility logic are defined in the TATR mapping cell above.

In [10]:
def evaluate_predictions(
    samples: List[Sample],
    predictions_by_image: Dict[int, List[Prediction]],
    config: EvalConfig,
) -> Dict[str, Any]:
    def to_float(value: Any) -> float:
        if value is None:
            return math.nan
        if isinstance(value, torch.Tensor):
            if value.numel() == 0:
                return math.nan
            return float(value.detach().cpu().item())
        try:
            return float(value)
        except Exception:
            return math.nan

    def to_float_tensor(values: List[float], shape: Tuple[int, ...]) -> torch.Tensor:
        if not values:
            return torch.zeros(shape, dtype=torch.float32)
        return torch.tensor(values, dtype=torch.float32)

    def to_int_tensor(values: List[int]) -> torch.Tensor:
        if not values:
            return torch.zeros((0,), dtype=torch.int64)
        return torch.tensor(values, dtype=torch.int64)

    def safe_div(numerator: float, denominator: float) -> float:
        return float(numerator / denominator) if denominator > 0 else 0.0

    def compute_detection_prf1(iou_threshold: float = 0.50) -> Dict[str, Any]:
        total_tp = 0
        total_fp = 0
        total_fn = 0

        per_class_counts: Dict[int, Dict[str, int]] = {
            class_id: {"tp": 0, "fp": 0, "fn": 0}
            for class_id in range(len(CLASS_NAMES))
        }

        for sample in samples:
            gt_by_class: Dict[int, List[List[float]]] = {c: [] for c in range(len(CLASS_NAMES))}
            pred_by_class: Dict[int, List[Tuple[float, List[float]]]] = {c: [] for c in range(len(CLASS_NAMES))}

            for gt_box, gt_label in zip(sample.gt_boxes_xyxy, sample.gt_labels):
                class_id = int(gt_label)
                if 0 <= class_id < len(CLASS_NAMES):
                    gt_by_class[class_id].append(gt_box)

            for pred in predictions_by_image.get(sample.image_id, []):
                class_id = int(pred.category_id)
                if 0 <= class_id < len(CLASS_NAMES):
                    pred_by_class[class_id].append((float(pred.score), pred.bbox_xyxy))

            for class_id in range(len(CLASS_NAMES)):
                gt_boxes = gt_by_class[class_id]
                class_preds = sorted(pred_by_class[class_id], key=lambda x: x[0], reverse=True)
                matched_gt: set[int] = set()

                tp = 0
                for _, pred_box in class_preds:
                    best_gt_idx = -1
                    best_iou = 0.0
                    for gt_idx, gt_box in enumerate(gt_boxes):
                        if gt_idx in matched_gt:
                            continue
                        iou = box_iou_xyxy(pred_box, gt_box)
                        if iou > best_iou:
                            best_iou = iou
                            best_gt_idx = gt_idx

                    if best_gt_idx >= 0 and best_iou >= iou_threshold:
                        matched_gt.add(best_gt_idx)
                        tp += 1

                fp = len(class_preds) - tp
                fn = len(gt_boxes) - tp

                per_class_counts[class_id]["tp"] += tp
                per_class_counts[class_id]["fp"] += fp
                per_class_counts[class_id]["fn"] += fn

                total_tp += tp
                total_fp += fp
                total_fn += fn

        precision = safe_div(total_tp, total_tp + total_fp)
        recall = safe_div(total_tp, total_tp + total_fn)
        f1 = safe_div(2 * precision * recall, precision + recall)

        per_class_prf1: Dict[int, Dict[str, float]] = {}
        for class_id in range(len(CLASS_NAMES)):
            tp = per_class_counts[class_id]["tp"]
            fp = per_class_counts[class_id]["fp"]
            fn = per_class_counts[class_id]["fn"]
            p = safe_div(tp, tp + fp)
            r = safe_div(tp, tp + fn)
            per_class_prf1[class_id] = {
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "precision": p,
                "recall": r,
                "f1": safe_div(2 * p * r, p + r),
            }

        return {
            "overall": {
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "tp": total_tp,
                "fp": total_fp,
                "fn": total_fn,
            },
            "per_class": per_class_prf1,
        }

    preds: List[Dict[str, torch.Tensor]] = []
    targets: List[Dict[str, torch.Tensor]] = []

    for sample in samples:
        targets.append(
            {
                "boxes": to_float_tensor(sample.gt_boxes_xyxy, (0, 4)).reshape(-1, 4),
                "labels": to_int_tensor(sample.gt_labels),
            }
        )

        sample_preds = predictions_by_image.get(sample.image_id, [])
        preds.append(
            {
                "boxes": to_float_tensor([p.bbox_xyxy for p in sample_preds], (0, 4)).reshape(-1, 4),
                "scores": to_float_tensor([float(p.score) for p in sample_preds], (0,)),
                "labels": to_int_tensor([int(p.category_id) for p in sample_preds]),
            }
        )

    def compute_map(iou_thresholds: Sequence[float]) -> Dict[str, Any]:
        metric = MeanAveragePrecision(
            box_format="xyxy",
            iou_type="bbox",
            class_metrics=True,
            iou_thresholds=[float(t) for t in iou_thresholds],
        )
        metric.update(preds, targets)
        return metric.compute()

    def get_per_class(output: Dict[str, Any], key: str) -> Dict[int, float]:
        classes = output.get("classes")
        values = output.get(key)
        if classes is None or values is None:
            return {}

        classes_list = classes.detach().cpu().tolist() if isinstance(classes, torch.Tensor) else list(classes)
        values_list = values.detach().cpu().tolist() if isinstance(values, torch.Tensor) else list(values)
        return {int(class_id): float(value) for class_id, value in zip(classes_list, values_list)}

    full_metrics = compute_map(config.map_thresholds)

    ap_by_threshold: Dict[float, float] = {}
    per_class_ap_by_threshold: Dict[str, Dict[str, float]] = {class_name: {} for class_name in CLASS_NAMES}

    for threshold in config.map_thresholds:
        threshold_value = float(threshold)
        threshold_metrics = compute_map([threshold_value])

        ap_by_threshold[threshold_value] = to_float(threshold_metrics.get("map"))
        threshold_per_class = get_per_class(threshold_metrics, "map_per_class")

        for class_id, class_name in enumerate(CLASS_NAMES):
            per_class_ap_by_threshold[class_name][f"{threshold_value:.2f}"] = threshold_per_class.get(
                class_id,
                math.nan,
            )

    per_class_map_50_95_by_id = get_per_class(full_metrics, "map_per_class")
    per_class_mar_100_by_id = get_per_class(full_metrics, "mar_100_per_class")
    prf1_stats = compute_detection_prf1(iou_threshold=0.50)

    class_rows: List[Dict[str, Any]] = []
    for class_id, class_name in enumerate(CLASS_NAMES):
        class_prf1 = prf1_stats["per_class"].get(class_id, {})
        class_rows.append(
            {
                "class_id": class_id,
                "class_name": class_name,
                "AP@0.50": per_class_ap_by_threshold[class_name].get("0.50", math.nan),
                "mAP@0.50:0.95": per_class_map_50_95_by_id.get(class_id, math.nan),
                "mAR@100": per_class_mar_100_by_id.get(class_id, math.nan),
                "precision": class_prf1.get("precision", 0.0),
                "recall": class_prf1.get("recall", 0.0),
                "f1": class_prf1.get("f1", 0.0),
                "tp": class_prf1.get("tp", 0),
                "fp": class_prf1.get("fp", 0),
                "fn": class_prf1.get("fn", 0),
            }
        )

    class_df = pd.DataFrame(class_rows)

    map_50 = to_float(full_metrics.get("map_50"))
    if not np.isfinite(map_50):
        map_50 = ap_by_threshold.get(0.50, math.nan)

    summary = {
        "mAP@0.50": map_50,
        "mAP@0.50:0.95": to_float(full_metrics.get("map")),
        "mAP@0.75": to_float(full_metrics.get("map_75")),
        "mAR@1": to_float(full_metrics.get("mar_1")),
        "mAR@10": to_float(full_metrics.get("mar_10")),
        "mAR@100": to_float(full_metrics.get("mar_100")),
        "precision": prf1_stats["overall"]["precision"],
        "recall": prf1_stats["overall"]["recall"],
        "f1": prf1_stats["overall"]["f1"],
    }

    per_class_map_50_95 = {
        CLASS_NAMES[class_id]: per_class_map_50_95_by_id.get(class_id, math.nan)
        for class_id in range(len(CLASS_NAMES))
    }

    per_class_prf1 = {
        CLASS_NAMES[class_id]: {
            "precision": prf1_stats["per_class"][class_id]["precision"],
            "recall": prf1_stats["per_class"][class_id]["recall"],
            "f1": prf1_stats["per_class"][class_id]["f1"],
            "tp": prf1_stats["per_class"][class_id]["tp"],
            "fp": prf1_stats["per_class"][class_id]["fp"],
            "fn": prf1_stats["per_class"][class_id]["fn"],
        }
        for class_id in range(len(CLASS_NAMES))
    }

    return {
        "summary": summary,
        "class_metrics": class_df,
        "ap_by_threshold": ap_by_threshold,
        "per_class_map_50_95": per_class_map_50_95,
        "per_class_ap_by_threshold": per_class_ap_by_threshold,
        "per_class_prf1": per_class_prf1,
    }

## Export

Serialize predictions and metrics to JSON/CSV outputs, and optionally save visualization overlays.

In [11]:
def save_predictions_json(
    samples: List[Sample],
    predictions_by_image: Dict[int, List[Prediction]],
    out_path: Path,
) -> None:
    payload = []
    for s in samples:
        preds = predictions_by_image.get(s.image_id, [])
        payload.append(
            {
                "image_id": s.image_id,
                "file_name": s.file_name,
                "predictions": [
                    {
                        "bbox_xyxy": p.bbox_xyxy,
                        "bbox_xywh": xyxy_to_coco_xywh(p.bbox_xyxy),
                        "category_id": p.category_id,
                        "category_name": CLASS_NAMES[p.category_id],
                        "score": p.score,
                        "source": p.source,
                    }
                    for p in preds
                ],
            }
        )
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)


def save_visualizations(
    samples: List[Sample],
    predictions_by_image: Dict[int, List[Prediction]],
    out_dir: Path,
    max_images: int,
) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    for s in samples[:max_images]:
        gt_img = s.image.copy()
        gt_draw = ImageDraw.Draw(gt_img)
        for box, class_id in zip(s.gt_boxes_xyxy, s.gt_labels):
            x1, y1, x2, y2 = box
            gt_draw.rectangle((x1, y1, x2, y2), outline="green", width=2)
            gt_draw.text((x1 + 2, y1 + 2), f"GT:{CLASS_NAMES[class_id]}", fill="green")

        pred_img = s.image.copy()
        pred_draw = ImageDraw.Draw(pred_img)
        for pred in predictions_by_image.get(s.image_id, []):
            x1, y1, x2, y2 = pred.bbox_xyxy
            pred_draw.rectangle((x1, y1, x2, y2), outline="red", width=2)
            pred_draw.text(
                (x1 + 2, y1 + 2),
                f"P:{CLASS_NAMES[pred.category_id]}:{pred.score:.2f}",
                fill="red",
            )

        gt_img.save(out_dir / f"{s.image_id:06d}_gt.png")
        pred_img.save(out_dir / f"{s.image_id:06d}_pred.png")

## Pipeline Orchestration

`run_experiment` is the main entry point. It handles end-to-end flow and writes JSON/CSV artifacts for quick inspection.

In [12]:
def run_experiment(config: EvalConfig) -> Dict[str, Any]:
    np.random.seed(config.random_seed)
    random.seed(config.random_seed)
    torch.manual_seed(config.random_seed)

    output_dir = Path(config.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    with (output_dir / "config.json").open("w", encoding="utf-8") as f:
        json.dump(asdict(config), f, indent=2)

    samples = load_samples(config)
    model_bundle = load_tatr(config)

    predictions_by_image = run_tatr_inference(
        samples=samples,
        config=config,
        model_bundle=model_bundle,
    )

    eval_out = evaluate_predictions(samples, predictions_by_image, config)
    summary = eval_out["summary"]
    class_df: pd.DataFrame = eval_out["class_metrics"]

    save_predictions_json(samples, predictions_by_image, output_dir / "predictions.json")
    class_df.to_csv(output_dir / "class_metrics.csv", index=False)

    with (output_dir / "summary.json").open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    with (output_dir / "map_thresholds.json").open("w", encoding="utf-8") as f:
        json.dump(eval_out["ap_by_threshold"], f, indent=2)
    with (output_dir / "per_class_map_50_95.json").open("w", encoding="utf-8") as f:
        json.dump(eval_out["per_class_map_50_95"], f, indent=2)
    with (output_dir / "per_class_ap_by_threshold.json").open("w", encoding="utf-8") as f:
        json.dump(eval_out["per_class_ap_by_threshold"], f, indent=2)
    with (output_dir / "per_class_prf1.json").open("w", encoding="utf-8") as f:
        json.dump(eval_out["per_class_prf1"], f, indent=2)

    if config.save_visualizations:
        save_visualizations(
            samples=samples,
            predictions_by_image=predictions_by_image,
            out_dir=output_dir / "visualizations",
            max_images=config.num_visualizations,
        )

    print("Summary metrics:")
    print(json.dumps(summary, indent=2))
    print("\nPer-class mAP@0.50:0.95:")
    print(json.dumps(eval_out["per_class_map_50_95"], indent=2))
    print("\nPer-class precision/recall/F1:")
    print(json.dumps(eval_out["per_class_prf1"], indent=2))
    print("\nClass metrics:")
    print(class_df)

    return {
        "config": config,
        "samples": samples,
        "predictions_by_image": predictions_by_image,
        "evaluation": eval_out,
    }

In [18]:
model_bundle = load_tatr(CONFIG)

print("Detection backend:", CONFIG.detection_backend)
print("Structure labels:", model_bundle["structure_model"].config.id2label)
if model_bundle.get("detection_model") is not None:
    print("Detection labels:", model_bundle["detection_model"].config.id2label)
if model_bundle.get("layout_predictor") is not None:
    print("Surya layout predictor loaded")
if model_bundle.get("yolo_detector") is not None:
    print("YOLO detector loaded from:", CONFIG.yolo_model_path)

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Detection backend: yolo
Structure labels: {0: 'table', 1: 'table column', 2: 'table row', 3: 'table column header', 4: 'table projected row header', 5: 'table spanning cell'}
YOLO detector loaded from: best.pt


## Run the Demo

This cell executes the full pipeline:
1. load samples
2. run TATR inference
3. compute TorchMetrics detection metrics
4. save outputs under `CONFIG.output_dir`.

In [16]:
CONFIG.max_samples = 500
CONFIG.detection_backend = "yolo"
CONFIG.yolo_model_path = "best.pt"   # or full path
CONFIG.yolo_conf_threshold = 0.25
CONFIG.yolo_table_class_names = ("table",)  # adjust if your class name differs
results = run_experiment(CONFIG)

Loading HF dataset:   0%|          | 0/500 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Inference chunks:   0%|          | 0/5 [00:00<?, ?it/s]

Summary metrics:
{
  "mAP@0.50": 0.8012109398841858,
  "mAP@0.50:0.95": 0.6502954363822937,
  "mAP@0.75": 0.7455725073814392,
  "mAR@1": 0.3361046314239502,
  "mAR@10": 0.6379757523536682,
  "mAR@100": 0.6884035468101501,
  "precision": 0.9459396930670163,
  "recall": 0.8301861252115059,
  "f1": 0.8842909667651936
}

Per-class mAP@0.50:0.95:
{
  "table": 0.8252741694450378,
  "table column": 0.7662197947502136,
  "table row": 0.6128768920898438,
  "table column header": 0.6417714953422546,
  "table projected row header": 0.5496528148651123,
  "table spanning cell": 0.5059776306152344
}

Per-class precision/recall/F1:
{
  "table": {
    "precision": 0.9979919678714859,
    "recall": 0.8338926174496645,
    "f1": 0.9085923217550276,
    "tp": 497,
    "fp": 1,
    "fn": 99
  },
  "table column": {
    "precision": 0.9777034559643255,
    "recall": 0.8245064243183955,
    "f1": 0.8945936756205373,
    "tp": 2631,
    "fp": 60,
    "fn": 560
  },
  "table row": {
    "precision": 0.9598036

In [15]:
# # Optional: use the same 500-sample subset as prior Surya experiment docs.
# CONFIG.max_samples = 500
#
# # Run end-to-end evaluation.
# results = run_experiment(CONFIG)

Loading HF dataset:   0%|          | 0/500 [00:00<?, ?it/s]

Some weights of the model checkpoint at microsoft/table-transformer-detection were not used when initializing TableTransformerForObjectDetection: ['model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing TableTransformerForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TableTransformerForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Inference chunks:   0%|          | 0/5 [00:00<?, ?it/s]

Summary metrics:
{
  "mAP@0.50": 0.7775396108627319,
  "mAP@0.50:0.95": 0.6349605321884155,
  "mAP@0.75": 0.727209746837616,
  "mAR@1": 0.3298945128917694,
  "mAR@10": 0.6237223744392395,
  "mAR@100": 0.6735424399375916,
  "precision": 0.946349905601007,
  "recall": 0.8142131979695432,
  "f1": 0.8753228799068651
}

Per-class mAP@0.50:0.95:
{
  "table": 0.8249801993370056,
  "table column": 0.7697574496269226,
  "table row": 0.6040967106819153,
  "table column header": 0.6079044342041016,
  "table projected row header": 0.5133227109909058,
  "table spanning cell": 0.48970162868499756
}

Per-class precision/recall/F1:
{
  "table": {
    "precision": 1.0,
    "recall": 0.837248322147651,
    "f1": 0.9114155251141552,
    "tp": 499,
    "fp": 0,
    "fn": 97
  },
  "table column": {
    "precision": 0.9782848730217151,
    "recall": 0.83296772171733,
    "f1": 0.899796885578876,
    "tp": 2658,
    "fp": 59,
    "fn": 533
  },
  "table row": {
    "precision": 0.9641635891122965,
    "reca

In [17]:
summary = results["evaluation"]["summary"]
print(f"mAP@0.50: {summary['mAP@0.50']:.6f}")
print(f"mAP@0.50:0.95: {summary['mAP@0.50:0.95']:.6f}")
print(f"mAP@0.75: {summary['mAP@0.75']:.6f}")
print(f"mAR@1: {summary['mAR@1']:.6f}")
print(f"mAR@10: {summary['mAR@10']:.6f}")
print(f"mAR@100: {summary['mAR@100']:.6f}")
print(f"Precision: {summary['precision']:.6f}")
print(f"Recall: {summary['recall']:.6f}")
print(f"F1: {summary['f1']:.6f}")

results["evaluation"]["class_metrics"][[
    "class_name",
    "precision",
    "recall",
    "f1",
    "tp",
    "fp",
    "fn",
    "AP@0.50",
    "mAP@0.50:0.95",
]]

mAP@0.50: 0.801211
mAP@0.50:0.95: 0.650295
mAP@0.75: 0.745573
mAR@1: 0.336105
mAR@10: 0.637976
mAR@100: 0.688404
Precision: 0.945940
Recall: 0.830186
F1: 0.884291


,class_name,precision,recall,f1,tp,fp,fn,AP@0.50,mAP@0.50:0.95
0,table,0.997992,0.833893,0.908592,497,1,99,0.831484,0.825274
1,table column,0.977703,0.824506,0.894594,2631,60,560,0.821460,0.766220
2,table row,0.959804,0.851878,0.902626,7235,303,1258,0.850254,0.612877
3,table column header,0.953815,0.813356,0.878004,475,23,109,0.804408,0.641771
4,table projected row header,0.874486,0.814176,0.843254,425,61,97,0.800574,0.549653
5,table spanning cell,0.798567,0.722102,0.758412,1003,253,386,0.699086,0.505978
